<a href="https://colab.research.google.com/github/EduRubira2/Rubira_lib/blob/main/Newton_Raphson.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Nome: <font color="red">**Eduardo Nunes Rubira**</font>\
Matrícula:<font color="red"> **169555**</font>\
Método: <font color="red">**Método do Newton Raphson**</font>


Resumo: ***Método do Newton Raphson funciona utilizando a derivada que cria retas tangentes a fim de encontrar as raízes, onde ela traça a reta tangente na função, após isso ela volta para a função fazendo uma nova iteração e criando uma nova reta tangente até se aproximar da raiz, pode ser deduzido por conta que esse método cria triângulos retangulos onde a cada iteração é um novo triangulo retangulo $$f'(x) = \frac{f(x_i) - o}{x_i - x_i + 1}$$***

In [ ]:
"""
================================================================================
MÉTODO DE NEWTON-RAPHSON — Raízes de f(x) = 0
================================================================================
Disciplina : Métodos Numéricos
Autor      : Seu Nome
Matrícula  : 000000
Data       : 2026-04-28
================================================================================

DESCRIÇÃO DO PROBLEMA:
    Dada uma equação f(x) = 0, o método usa a tangente à curva em x_n
    para estimar onde ela cruza o eixo x, gerando x_(n+1):

        x_(n+1) = x_n - f(x_n) / f'(x_n)

    Exemplo clássico:
        f(x)  = e^(-x) - x
        f'(x) = -e^(-x) - 1

MÉTODO UTILIZADO:
    Newton-Raphson — convergência quadrática (muito mais rápido que bissecção
    e ponto fixo), mas exige que f'(x) ≠ 0 e um bom chute inicial x0.

DIFERENÇA PARA O PONTO FIXO:
    Ponto Fixo   → você escolhe g(x) manualmente
    Newton-Raphson → g(x) é sempre x - f(x)/f'(x), gerada automaticamente

ENTRADAS:
    f_expr          → string com a expressão de f(x)
    df_expr         → string com a expressão de f'(x) (derivada)
    x0              → chute inicial
    tol             → tolerância do erro relativo percentual (%)
    max_iter        → número máximo de iterações
    raiz_verdadeira → raiz analítica (opcional, para calcular erro real)

SAÍDAS:
    Obrigatórias:
        x1        → aproximação da raiz encontrada
        iterações → número de iterações realizadas
        aviso     → mensagem se não convergir

    Opcionais (todas implementadas):
        1. f(x1)  → valor da função na raiz aproximada
        2. Tabela → x1, f(x1), f'(x1) e erro ao longo das iterações
        3. Gráfico do erro relativo por iteração
        4. Gráfico de f(x) destacando a raiz aproximada
        +. [Extra] Gráfico geométrico com as tangentes de cada iteração
================================================================================
"""

# ==============================================================================
# 1. IMPORTS
# ==============================================================================
import math                        # funções matemáticas nativas do Python (exp, sin, log…)
import numpy as np                 # numpy: operações numéricas vetorizadas; apelido "np" por convenção
import matplotlib.pyplot as plt    # pyplot: módulo de criação de gráficos; apelido "plt" por convenção
import pandas as pd                # pandas: manipulação de tabelas (DataFrame); apelido "pd" por convenção


# ==============================================================================
# 2. FUNÇÕES AUXILIARES
# ==============================================================================

def criar_funcao(expr):
    """
    Compila uma string digitada pelo usuário em uma função Python segura.

    Parâmetros
    ----------
    expr : str
        Expressão matemática em função de x.
        Pode usar: math.exp, math.sin, math.cos, math.log, np.*, etc.
        Exemplos:
            f(x)  → "math.exp(-x) - x"
            f'(x) → "-math.exp(-x) - 1"

    Retorna
    -------
    callable
        Função avaliável numericamente: f(x) ou f'(x).
    """
    def func(x):
        try:                                                    # tenta calcular; se der erro não trava o programa
            return eval(expr, {"x": x, "math": math, "np": np})  # eval() executa a string como código Python; disponibiliza x, math e np
        except Exception:                                       # captura qualquer erro de cálculo (divisão por zero, log negativo…)
            return float("nan")                                 # retorna NaN para pontos onde a função não é definida
    return func                                                 # devolve a função pronta para ser chamada como f(x) ou df(x)


# ==============================================================================
# 3. IMPLEMENTAÇÃO DO MÉTODO
# ==============================================================================

def newton_raphson(f, df, x0, tol, max_iter, raiz_verdadeira=None):
    """
    Método de Newton-Raphson: itera x_(n+1) = x_n - f(x_n)/f'(x_n).

    Geometricamente: traça a reta tangente à curva em x_n e usa o ponto
    onde ela cruza o eixo x como próxima estimativa.

    Parâmetros
    ----------
    f               : callable → função f(x)
    df              : callable → derivada f'(x)
    x0              : float    → chute inicial
    tol             : float    → tolerância do erro relativo percentual (%)
    max_iter        : int      → número máximo de iterações
    raiz_verdadeira : float | None → raiz analítica para cálculo do erro real

    Retorna
    -------
    x1      : float      → última aproximação calculada
    tabela  : list[dict] → histórico completo das iterações

    Levanta
    -------
    ZeroDivisionError
        Se f'(x_n) = 0, a tangente é horizontal e o método não consegue avançar.
    """

    tabela = []                    # lista vazia para guardar os dados de cada iteração

    for i in range(max_iter):      # loop limitado a max_iter iterações para evitar loop infinito

        fx  = f(x0)                # avalia f no ponto atual
        dfx = df(x0)               # avalia a derivada no ponto atual (inclinação da tangente)

        # --- Verificação de derivada nula (tangente horizontal = sem cruzamento) ---
        if dfx == 0 or math.isnan(dfx):
            raise ZeroDivisionError(
                f"Derivada nula ou inválida em x = {x0:.6f}. "
                "Escolha outro x0 ou revise f'(x)."
            )

        # --- Passo principal do método: fórmula de Newton-Raphson ---
        x1 = x0 - fx / dfx         # nova estimativa = interseção da tangente com o eixo x

        # --- Erro relativo aproximado ---
        erro_aprox = abs((x1 - x0) / x1) * 100 if x1 != 0 else float("inf")  # erro relativo percentual entre estimativa nova e antiga

        # --- Erro real (só calculável se a raiz verdadeira for conhecida) ---
        erro_real = (
            abs((raiz_verdadeira - x1) / raiz_verdadeira) * 100
            if raiz_verdadeira is not None
            else float("nan")
        )

        # --- Salva dados desta iteração num dicionário e adiciona à lista ---
        tabela.append({
            "iteração":    i,
            "x_n":         x0,
            "f(x_n)":      fx,
            "f'(x_n)":     dfx,
            "x_(n+1)":     x1,
            "εa (%)":      erro_aprox,
            "erro real (%)": erro_real,
        })

        # --- Critério de parada: convergiu → sai do loop com break ---
        if erro_aprox < tol:
            break

        x0 = x1                    # atualiza a estimativa: x novo vira o x atual para a próxima iteração

    else:
        # bloco else do for: só executa se o loop acabou SEM break (não convergiu)
        print("Aviso: número máximo de iterações atingido sem convergência.")

    return x1, tabela              # retorna a melhor estimativa encontrada e o histórico completo


# ==============================================================================
# 4. SAÍDA / RELATÓRIO
# ==============================================================================

def imprimir_tabela(tabela, raiz_verdadeira):
    """
    Exibe o histórico de iterações como DataFrame formatado.

    Parâmetros
    ----------
    tabela          : list[dict]
    raiz_verdadeira : float | None
    """
    df = pd.DataFrame(tabela)      # converte a lista de dicionários numa tabela pandas (DataFrame)

    # remove a coluna de erro real se não foi fornecida raiz verdadeira
    if raiz_verdadeira is None:
        df = df.drop(columns=["erro real (%)"])   # remove coluna do DataFrame para não exibir coluna cheia de NaN

    pd.set_option("display.float_format", "{:.6f}".format)  # configura o pandas para exibir 6 casas decimais em todos os floats
    print(df.to_string(index=False))                         # imprime a tabela sem a coluna de índice (0,1,2…) do pandas
    return df


# ==============================================================================
# 5. VISUALIZAÇÃO
# ==============================================================================

def plotar_resultados(tabela, raiz_verdadeira, tol):
    """
    [Saídas opcionais 2 e 3] Gera dois gráficos lado a lado:
      - Convergência de x_(n+1) ao longo das iterações
      - Erro relativo percentual por iteração

    Parâmetros
    ----------
    tabela          : list[dict] → histórico de iterações
    raiz_verdadeira : float | None
    tol             : float → tolerância usada (para linha de referência)
    """

    # --- Extrair colunas da tabela com list comprehension ---
    iteracoes   = [r["iteração"]      for r in tabela]   # list comprehension: extrai a coluna de todos os dicionários
    valores_x   = [r["x_(n+1)"]      for r in tabela]   # estimativas x_(n+1) de cada iteração
    erros_reais = [r["erro real (%)"] for r in tabela]   # erros reais de cada iteração

    # --- Decide quantos painéis criar ---
    n_graficos = 2 if raiz_verdadeira is not None else 1  # decide quantos painéis criar dependendo se há raiz verdadeira
    fig, axs = plt.subplots(1, n_graficos, figsize=(6 * n_graficos, 5))  # cria figura com N painéis; figsize em polegadas
    fig.suptitle("Newton-Raphson — Convergência", fontsize=13, fontweight="bold")

    if n_graficos == 1:
        axs = [axs]                # quando há 1 painel, axs não é lista — forçamos ser lista para o código funcionar igual

    # --- Gráfico 1: Convergência de x ---
    axs[0].plot(iteracoes, valores_x, marker="o", color="royalblue",
                linewidth=2, markersize=5, label="x_(n+1)")
    if raiz_verdadeira is not None:
        axs[0].axhline(raiz_verdadeira, color="crimson", linestyle="--",   # linha horizontal no valor da raiz verdadeira
                       linewidth=1.2, label=f"Raiz verdadeira = {raiz_verdadeira}")
    axs[0].set_title("Convergência de x")
    axs[0].set_xlabel("Iteração")
    axs[0].set_ylabel("x")
    axs[0].legend()
    axs[0].grid(True, alpha=0.3)

    # --- Gráfico 2: Erro real por iteração (só se raiz verdadeira foi informada) ---
    if raiz_verdadeira is not None:
        axs[1].plot(iteracoes, erros_reais, marker="x", color="darkorange",
                    linewidth=2, markersize=6, label="Erro real (%)")
        axs[1].axhline(tol, color="gray", linestyle="--",                  # linha de referência da tolerância
                       linewidth=1, label=f"Tolerância = {tol}%")
        axs[1].set_title("Erro Real por Iteração")
        axs[1].set_xlabel("Iteração")
        axs[1].set_ylabel("Erro real (%)")
        axs[1].legend()
        axs[1].grid(True, alpha=0.3)

    plt.tight_layout()                                    # ajusta automaticamente o espaçamento entre painéis
    plt.savefig("newton_raphson_convergencia.png", dpi=150)  # salva o gráfico em PNG com resolução de 150 dpi
    plt.show()


def plotar_funcao(f, raiz, x0_original):
    """
    [Saída opcional 4 — enunciado] Gráfico de f(x) destacando a raiz aproximada.

    Mostra a curva completa de f(x) no intervalo relevante, o eixo x
    e um marcador vermelho na raiz encontrada pelo método.

    Parâmetros
    ----------
    f           : callable → função f(x)
    raiz        : float    → raiz aproximada encontrada
    x0_original : float    → chute inicial (para dimensionar o eixo x)
    """

    # --- Janela de visualização em torno da raiz ---
    margem = max(abs(raiz - x0_original) * 1.5, 1.0)   # define o "zoom": 1.5x a distância entre chute e raiz, mínimo 1
    x_min  = raiz - margem
    x_max  = raiz + margem
    x_vals = np.linspace(x_min, x_max, 400)             # 400 pontos igualmente espaçados para curva suave

    # --- Avaliar f em cada ponto ---
    f_vals = np.array([f(x) for x in x_vals], dtype=float)  # dtype=float garante que NaN seja tratado corretamente

    fig, ax = plt.subplots(figsize=(8, 5))
    fig.suptitle("Newton-Raphson — f(x) e raiz aproximada",
                 fontsize=13, fontweight="bold")

    # --- Curva de f(x) ---
    ax.plot(x_vals, f_vals, color="royalblue", linewidth=2, label="f(x)", zorder=3)

    # --- Eixo x (onde f(x) = 0) ---
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")   # linha y = 0 como referência

    # --- Raiz aproximada marcada ---
    ax.scatter([raiz], [f(raiz)], color="crimson", zorder=5, s=100,   # ponto vermelho sobre a curva na raiz
               label=f"Raiz ≈ {raiz:.6f}")
    ax.axvline(raiz, color="crimson", linewidth=0.8,                   # linha vertical tracejada na raiz
               linestyle=":", alpha=0.6)

    ax.set_xlabel("x")
    ax.set_ylabel("f(x)")
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()                                        # ajusta o espaçamento para não sobrepor elementos
    plt.savefig("newton_raphson_funcao.png", dpi=150)         # salva o gráfico em PNG
    plt.show()


def plotar_tangentes(f, raiz, x0_original, tabela):
    """
    [Saída extra — interpretação geométrica] Gráfico de f(x) com as retas
    tangentes de cada iteração desenhadas, mostrando como o método converge.

    Cada tangente parte de (x_n, f(x_n)) e cruza o eixo x em x_(n+1),
    que vira o ponto de partida da próxima tangente.

    Parâmetros
    ----------
    f           : callable   → função f(x)
    raiz        : float      → raiz aproximada encontrada
    x0_original : float      → chute inicial (para dimensionar o eixo x)
    tabela      : list[dict] → histórico de iterações (contém x_n e f'(x_n))
    """

    # --- Janela de visualização em torno da raiz ---
    margem = max(abs(raiz - x0_original) * 1.5, 1.0)   # define o "zoom": 1.5x a distância entre chute e raiz, mínimo 1
    x_min  = raiz - margem
    x_max  = raiz + margem
    x_vals = np.linspace(x_min, x_max, 400)             # 400 pontos igualmente espaçados para curva suave

    # --- Avaliar f em cada ponto ---
    f_vals = np.array([f(x) for x in x_vals], dtype=float)  # dtype=float garante que NaN seja tratado corretamente

    fig, ax = plt.subplots(figsize=(9, 6))
    fig.suptitle("Newton-Raphson — Interpretação Geométrica das Tangentes",
                 fontsize=13, fontweight="bold")

    # --- Curva de f(x) ---
    ax.plot(x_vals, f_vals, color="royalblue", linewidth=2.5, label="f(x)", zorder=3)
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")   # eixo x como referência

    # --- Desenhar tangente de cada iteração ---
    cores = plt.cm.Oranges(np.linspace(0.4, 0.9, len(tabela)))    # gradiente de cor para distinguir iterações

    for idx, linha in enumerate(tabela):
        xn    = linha["x_n"]        # ponto de partida da tangente nesta iteração
        fxn   = linha["f(x_n)"]     # valor de f no ponto de partida
        dfxn  = linha["f'(x_n)"]    # inclinação da tangente (derivada)
        x_next = linha["x_(n+1)"]   # onde a tangente cruza o eixo x

        # --- Reta tangente: y = f(xn) + f'(xn) * (x - xn) ---
        x_tang = np.linspace(min(xn, x_next) - 0.1, max(xn, x_next) + 0.1, 100)
        y_tang = fxn + dfxn * (x_tang - xn)                       # equação da reta tangente no ponto (xn, fxn)

        ax.plot(x_tang, y_tang, color=cores[idx], linewidth=1.2,
                linestyle="--", alpha=0.85)

        ax.scatter([xn], [fxn], color=cores[idx], zorder=5, s=50)  # marca o ponto de tangência na curva

        # seta vertical pontilhada do eixo x até (xn, f(xn)) — mostra de onde parte cada iteração
        ax.annotate("", xy=(xn, fxn), xytext=(xn, 0),
                    arrowprops=dict(arrowstyle="-", color=cores[idx],
                                   lw=0.8, linestyle="dotted"))

        # rótulo x0, x1, x2… alternando acima e abaixo para não sobrepor
        deslocamento = 0.05 if idx % 2 == 0 else -0.08
        y_range = ax.get_ylim()[1] - ax.get_ylim()[0]
        ax.text(xn, fxn + deslocamento * y_range,
                f"x{idx}", fontsize=8, color=cores[idx], ha="center")

    # --- Raiz final marcada ---
    ax.scatter([raiz], [f(raiz)], color="crimson", zorder=6, s=100,
               label=f"Raiz ≈ {raiz:.6f}")

    ax.set_xlabel("x")
    ax.set_ylabel("f(x)")
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()                                         # ajusta o espaçamento para não sobrepor elementos
    plt.savefig("newton_raphson_tangentes.png", dpi=150)       # salva o gráfico em PNG
    plt.show()


# ==============================================================================
# 6. PROGRAMA PRINCIPAL
# ==============================================================================

def main():
    print("=" * 65)
    print("  MÉTODO DE NEWTON-RAPHSON — Encontrar raiz de f(x) = 0")
    print("=" * 65)

    # --- Entradas do usuário ---
    try:                                                       # bloco protegido: se o usuário digitar algo inválido, capturamos o erro
        f_expr  = input("\nDigite f(x)   (ex: math.exp(-x) - x)      : ")
        df_expr = input("Digite f'(x)  (ex: -math.exp(-x) - 1)      : ")
        x0      = float(input("Valor inicial x₀                          : "))   # input() lê texto; float() converte — lança ValueError se falhar
        tol     = float(input("Tolerância percentual (ex: 5 para 5%)     : "))
        max_iter = int(input("Número máximo de iterações                : "))

        raiz_input      = input("Raiz verdadeira (Enter se não souber)     : ")
        raiz_verdadeira = float(raiz_input) if raiz_input.strip() else None      # converte se digitou algo, senão deixa None

    except ValueError as e:                                    # captura erro de conversão (ex: digitou "abc" no lugar de número)
        print(f"\nErro na entrada de dados: {e}")
        return

    print("\n" + "=" * 65)
    print(f"  f(x)  = {f_expr}")
    print(f"  f'(x) = {df_expr}")
    print(f"  x₀    = {x0}  |  Tolerância: {tol}%  |  Max iter: {max_iter}")
    print("=" * 65)

    # --- Transformar strings em funções Python chamáveis ---
    f  = criar_funcao(f_expr)    # transforma a string de f(x) em função chamável
    df = criar_funcao(df_expr)   # transforma a string de f'(x) em função chamável

    # --- Executar o método ---
    try:
        raiz, tabela = newton_raphson(f, df, x0, tol, max_iter, raiz_verdadeira)  # chama o método; desempacota os dois valores retornados
    except ZeroDivisionError as e:
        print(f"\nErro: {e}")
        return
    except Exception as e:
        print(f"\nErro durante a execução do método: {e}")
        return

    # --- [Opcional 2] Imprimir tabela ---
    print("\nHistórico de iterações:")
    print("-" * 65)
    imprimir_tabela(tabela, raiz_verdadeira)

    # --- Resultados finais (saídas obrigatórias + opcional 1) ---
    print("\n" + "=" * 65)
    print("  RESULTADOS")
    print("=" * 65)
    print(f"  [Obrigatório] Raiz encontrada : x1 ≈ {raiz:.8f}")
    print(f"  [Obrigatório] Iterações       : {len(tabela)}")
    print(f"  [Opcional 1]  f(x1)           : {f(raiz):.2e}  (deve ser ≈ 0)")
    print("=" * 65)

    # --- [Opcional 3] Gráfico de convergência e erro ---
    plotar_resultados(tabela, raiz_verdadeira, tol)

    # --- [Opcional 4 — enunciado] Gráfico de f(x) destacando a raiz ---
    plotar_funcao(f, raiz, x0)

    # --- [Extra] Gráfico geométrico com todas as tangentes ---
    plotar_tangentes(f, raiz, x0, tabela)


# ==============================================================================
# Ponto de entrada padrão Python
# ==============================================================================
if __name__ == "__main__":        # só executa main() se este arquivo for o ponto de entrada do programa
    main()

  MÉTODO DE NEWTON-RAPHSON — Encontrar raiz de f(x) = 0

Digite f(x)   (ex: math.exp(-x) - x)      : math.exp(-x) - x
Digite f'(x)  (ex: -math.exp(-x) - 1)      : -1 - math.exp(-x) - x
Valor inicial x₀                          : 0
